# KIS API 해외주식 테스트 노트북

모의투자 환경에서 해외주식(미국 ETF) API 기능을 하나씩 테스트합니다.

In [1]:
import asyncio
import httpx
import os
import time
from dotenv import load_dotenv

load_dotenv()

APP_KEY = os.getenv("KIS_APP_KEY", "")
APP_SECRET = os.getenv("KIS_APP_SECRET", "")
ACCOUNT = os.getenv("KIS_ACCOUNT_NUMBER", "50174429-01")
BASE_URL = "https://openapivts.koreainvestment.com:29443"  # 모의투자

CANO = ACCOUNT.split("-")[0]
ACNT_PRDT_CD = ACCOUNT.split("-")[1] if "-" in ACCOUNT else "01"

TOKEN = None

print(f"APP_KEY: {APP_KEY[:10]}...")
print(f"계좌: {CANO}-{ACNT_PRDT_CD}")
print(f"BASE_URL: {BASE_URL}")

APP_KEY: PSGZkLJQ2B...
계좌: 50174429-01
BASE_URL: https://openapivts.koreainvestment.com:29443


## 1. OAuth 토큰 발급

In [2]:
async def get_token():
    global TOKEN
    url = f"{BASE_URL}/oauth2/tokenP"
    body = {
        "grant_type": "client_credentials",
        "appkey": APP_KEY,
        "appsecret": APP_SECRET,
    }
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, json=body)
        data = resp.json()

    if "access_token" in data:
        TOKEN = data["access_token"]
        print(f"토큰 발급 성공 (유효: ~24시간)")
        print(f"토큰: {TOKEN[:30]}...")
    else:
        print(f"토큰 발급 실패: {data}")

await get_token()

토큰 발급 성공 (유효: ~24시간)
토큰: eyJ0eXAiOiJKV1QiLCJhbGciOiJIUz...


In [3]:
def headers(tr_id):
    """공통 헤더 생성"""
    return {
        "content-type": "application/json; charset=utf-8",
        "authorization": f"Bearer {TOKEN}",
        "appkey": APP_KEY,
        "appsecret": APP_SECRET,
        "tr_id": tr_id,
    }

print("헤더 함수 준비 완료")

헤더 함수 준비 완료


## 2. 해외주식 현재가 조회

In [4]:
async def get_price(ticker="QQQ", exchange="NAS"):
    """
    해외주식 현재가 조회
    exchange: NAS(나스닥), NYS(뉴욕), AMS(아멕스)
    """
    url = f"{BASE_URL}/uapi/overseas-price/v1/quotations/price"
    params = {"AUTH": "", "EXCD": exchange, "SYMB": ticker}

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(url, headers=headers("HHDFS76200200"), params=params)
        data = resp.json()

    if data.get("rt_cd") == "0":
        o = data["output"]
        print(f"{'종목':>8}: {o.get('rsym', ticker)}")
        print(f"{'현재가':>8}: ${float(o.get('last', 0)):,.2f}")
        print(f"{'전일종가':>8}: ${float(o.get('base', 0)):,.2f}")
        print(f"{'고가':>8}: ${float(o.get('high', 0)):,.2f}")
        print(f"{'저가':>8}: ${float(o.get('low', 0)):,.2f}")
        print(f"{'거래량':>8}: {o.get('tvol', '0')}")
        print(f"{'등락률':>8}: {o.get('rate', '0')}%")
        return float(o.get('last', 0))
    else:
        print(f"실패: {data.get('msg1')}")
        return None

qqq_price = await get_price("QQQ", "NAS")

      종목: DNASQQQ
     현재가: $597.26
    전일종가: $607.69
      고가: $604.14
      저가: $597.05
     거래량: 71836629
     등락률: 0%


In [5]:
# 여러 종목 한번에 조회
tickers = [
    ("QQQ", "NAS"),
    ("SPY", "AMS"),
    ("TQQQ", "NAS"),
    ("SOXX", "NAS"),
    ("GLD", "AMS"),
]

print(f"{'종목':<8} {'현재가':>10} {'등락률':>8}")
print("-" * 30)

for ticker, excd in tickers:
    url = f"{BASE_URL}/uapi/overseas-price/v1/quotations/price"
    params = {"AUTH": "", "EXCD": excd, "SYMB": ticker}
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(url, headers=headers("HHDFS76200200"), params=params)
        data = resp.json()
    if data.get("rt_cd") == "0":
        o = data["output"]
        print(f"{ticker:<8} ${float(o.get('last', 0)):>9,.2f} {o.get('rate', '0'):>7}%")
    else:
        print(f"{ticker:<8} 조회 실패: {data.get('msg1', '')}")
    await asyncio.sleep(0.05)  # rate limit

종목              현재가      등락률
------------------------------
QQQ      $   597.26       0%
SPY      $   666.06       0%
TQQQ     조회 실패: 초당 거래건수를 초과하였습니다.
SOXX     조회 실패: 초당 거래건수를 초과하였습니다.
GLD      $   466.88       0%


## 3. 해외주식 잔고 조회

In [6]:
async def get_balance():
    """해외주식 잔고 조회"""
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/inquire-balance"
    params = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": "NASD",
        "TR_CRCY_CD": "USD",
        "CTX_AREA_FK200": "",
        "CTX_AREA_NK200": "",
    }

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(url, headers=headers("VTTS3012R"), params=params)
        data = resp.json()

    if data.get("rt_cd") == "0":
        o2 = data.get("output2", {})
        holdings = data.get("output1", [])

        print("=== 계좌 요약 ===")
        print(f"  주문 가능 금액: ${float(o2.get('ovrs_ord_psbl_amt', 0)):,.2f}")
        print(f"  외화 매입금액: ${float(o2.get('frcr_pchs_amt1', 0)):,.2f}")
        print(f"  평가손익금액: ${float(o2.get('tot_evlu_pfls_amt', 0)):,.2f}")
        print()

        if holdings:
            print("=== 보유 종목 ===")
            for h in holdings:
                qty = float(h.get('ovrs_cblc_qty', 0))
                if qty > 0:
                    print(f"  {h.get('ovrs_pdno', '?'):>6} | "
                          f"{qty:>8.4f}주 | "
                          f"평균가 ${float(h.get('pchs_avg_pric', 0)):>9,.2f} | "
                          f"현재가 ${float(h.get('now_pric2', 0)):>9,.2f} | "
                          f"손익률 {h.get('evlu_pfls_rt', '0')}%")
        else:
            print("보유 종목 없음")

        return data
    else:
        print(f"실패: {data.get('msg1')}")
        return data

balance_data = await get_balance()

=== 계좌 요약 ===
  주문 가능 금액: $0.00
  외화 매입금액: $0.00
  평가손익금액: $0.00

보유 종목 없음


## 4. 주문 가능 금액 조회

In [12]:
async def get_buying_power(ticker="QQQ", exchange="NASD"):
    """해외주식 매수 가능 금액/수량 조회"""
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/inquire-psamount"
    params = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": exchange,
        "OVRS_ORD_UNPR": "0",
        "ITEM_CD": ticker,
    }

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(url, headers=headers("VTTS3007R"), params=params)
        data = resp.json()

    if data.get("rt_cd") == "0":
        o = data.get("output", {})
        print(f"=== {ticker} 매수 가능 조회 ===")
        print(f"  외화 주문 가능 금액: ${float(o.get('ovrs_ord_psbl_amt', 0)):,.2f}")
        print(f"  최대 매수 가능 수량: {o.get('max_buy_qty', '0')}주")
        print(f"  원화 환산: ₩{float(o.get('frcr_ord_psbl_amt1', 0)):,.0f}")
        return data
    else:
        print(f"실패 (rt_cd={data.get('rt_cd')}): {data.get('msg1')}")
        print(f"  -> 가상자금이 충전되지 않았을 수 있습니다.")
        return data

bp_data = await get_buying_power("QQQ")

=== QQQ 매수 가능 조회 ===
  외화 주문 가능 금액: $100,000.00
  최대 매수 가능 수량: 0주
  원화 환산: ₩269,149


## 5. 해외주식 매수 주문 테스트

**주의**: 가상자금이 있어야 주문이 체결됩니다. 없으면 주문 자체는 접수되지만 잔고 부족으로 실패합니다.

In [13]:
async def buy_order(ticker="QQQ", qty=1, price=0, exchange="NASD"):
    """
    해외주식 매수 주문
    price=0: 시장가, price>0: 지정가
    """
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order"
    body = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": exchange,
        "PDNO": ticker,
        "ORD_QTY": str(qty),
        "OVRS_ORD_UNPR": str(price),
        "ORD_SVR_DVSN_CD": "0",
        "ORD_DVSN": "00",
    }

    print(f"주문 요청: {ticker} {qty}주 @ {'시장가' if price == 0 else f'${price}'}")
    print(f"  tr_id: VTTT1002U (모의투자 매수)")
    print(f"  body: {body}")
    print()

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, headers=headers("VTTT1002U"), json=body)
        data = resp.json()

    if data.get("rt_cd") == "0":
        o = data.get("output", {})
        print(f"  주문 성공!")
        print(f"  주문번호: {o.get('ODNO', '?')}")
        print(f"  주문단가: {o.get('OVRS_ORD_UNPR', '?')}")
    else:
        print(f"  주문 실패: {data.get('msg1')}")
        if "잔고" in str(data.get('msg1', '')) or "금액" in str(data.get('msg1', '')):
            print(f"  -> 가상자금을 먼저 충전하세요!")

    return data

# QQQ 1주 시장가 매수 (가상자금 필요)
buy_result = await buy_order("QQQ", qty=1, price=0)
print("매수 테스트를 실행하려면 위 주석을 해제하세요.")

주문 요청: QQQ 1주 @ 시장가
  tr_id: VTTT1002U (모의투자 매수)
  body: {'CANO': '50174429', 'ACNT_PRDT_CD': '01', 'OVRS_EXCG_CD': 'NASD', 'PDNO': 'QQQ', 'ORD_QTY': '1', 'OVRS_ORD_UNPR': '0', 'ORD_SVR_DVSN_CD': '0', 'ORD_DVSN': '00'}

  주문 실패: 모의투자 장종료 입니다.
매수 테스트를 실행하려면 위 주석을 해제하세요.


## 6. 해외주식 매도 주문 테스트

In [14]:
async def sell_order(ticker="QQQ", qty=1, price=0, exchange="NASD"):
    """
    해외주식 매도 주문
    price=0: 시장가, price>0: 지정가
    """
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order"
    body = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": exchange,
        "PDNO": ticker,
        "ORD_QTY": str(qty),
        "OVRS_ORD_UNPR": str(price),
        "ORD_SVR_DVSN_CD": "0",
        "ORD_DVSN": "00",
    }

    print(f"매도 요청: {ticker} {qty}주 @ {'시장가' if price == 0 else f'${price}'}")

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, headers=headers("VTTT1006U"), json=body)
        data = resp.json()

    if data.get("rt_cd") == "0":
        o = data.get("output", {})
        print(f"  매도 성공! 주문번호: {o.get('ODNO', '?')}")
    else:
        print(f"  매도 실패: {data.get('msg1')}")

    return data

# QQQ 1주 시장가 매도 (보유종목 필요)
sell_result = await sell_order("QQQ", qty=1, price=0)
print("매도 테스트를 실행하려면 위 주석을 해제하세요.")

매도 요청: QQQ 1주 @ 시장가
  매도 실패: 모의투자에서는 해당업무가 제공되지 않습니다.
매도 테스트를 실행하려면 위 주석을 해제하세요.


## 7. 체결 내역 조회

In [15]:
from datetime import datetime, timedelta

async def get_executions(days=7):
    """해외주식 체결 내역 조회"""
    today = datetime.now()
    start = (today - timedelta(days=days)).strftime("%Y%m%d")
    end = today.strftime("%Y%m%d")

    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/inquire-ccnl"
    params = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "PDNO": "",
        "ORD_STRT_DT": start,
        "ORD_END_DT": end,
        "SLL_BUY_DVSN": "00",
        "CCLD_NCCS_DVSN": "00",
        "OVRS_EXCG_CD": "NASD",
        "SORT_SQN": "DS",
        "ORD_GNO_BRNO": "",
        "ODNO": "",
        "CTX_AREA_FK200": "",
        "CTX_AREA_NK200": "",
    }

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(url, headers=headers("VTTS3018R"), params=params)
        data = resp.json()

    if data.get("rt_cd") == "0":
        orders = data.get("output", [])
        if orders:
            print(f"=== 최근 {days}일 체결 내역 ({len(orders)}건) ===")
            for o in orders[:10]:
                side = "매수" if o.get("sll_buy_dvsn_cd") == "02" else "매도"
                print(f"  {o.get('ord_dt', '?')} | {side} | "
                      f"{o.get('pdno', '?')} | "
                      f"{o.get('ft_ccld_qty', '0')}주 @ "
                      f"${float(o.get('ft_ccld_unpr3', 0)):,.2f} | "
                      f"주문번호: {o.get('odno', '?')}")
        else:
            print(f"최근 {days}일 체결 내역 없음")
    else:
        print(f"실패: {data.get('msg1')}")

    return data

exec_data = await get_executions()

최근 7일 체결 내역 없음


## 8. 예약주문 테스트 (소수점 매매 + 모의투자 매도)

일반 주문 API 한계:
- **소수점 매수**: ORD_QTY에 소수점 → "MCA 전문바디 구성 중 오류" ❌
- **모의투자 매도**: VTTT1006U → "해당업무가 제공되지 않습니다" ❌

**예약주문 API**는 모의투자 매도(VTTT3016U)를 지원하고, `FT_ORD_QTY` 파라미터가 있어 소수점이 가능할 수 있습니다.

In [16]:
async def resv_buy_order(ticker="QQQ", qty=0.5, price=600.0, exchange="NASD"):
    """
    해외주식 예약주문 매수 (모의투자 지원, 소수점 테스트)
    tr_id: VTTT3014U (모의투자 예약매수)
    FT_ORD_QTY: 소수점 수량 가능 여부 테스트
    """
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order-resv"
    body = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": exchange,
        "PDNO": ticker,
        "FT_ORD_QTY": str(qty),
        "FT_ORD_UNPR3": str(price),
        "ORD_SVR_DVSN_CD": "0",
    }

    print(f"예약 매수 테스트: {ticker} {qty}주 @ ${price}")
    print(f"  tr_id: VTTT3014U (모의투자 예약매수)")
    print(f"  FT_ORD_QTY = \"{qty}\"")
    print(f"  body: {body}")
    print()

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, headers=headers("VTTT3014U"), json=body)
        data = resp.json()

    rt_cd = data.get("rt_cd")
    msg = data.get("msg1", "")

    if rt_cd == "0":
        o = data.get("output", {})
        print(f"  예약 매수 성공!")
        print(f"  -> 소수점 매매 지원 확인!" if qty != int(qty) else "  -> 정수 주문 성공")
        print(f"  주문번호: {o}")
    else:
        print(f"  결과 (rt_cd={rt_cd}): {msg}")

    return data


async def resv_sell_order(ticker="QQQ", qty=1, price=600.0, exchange="NASD"):
    """
    해외주식 예약주문 매도 (모의투자 지원!)
    tr_id: VTTT3016U (모의투자 예약매도)
    """
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order-resv"
    body = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": exchange,
        "PDNO": ticker,
        "FT_ORD_QTY": str(qty),
        "FT_ORD_UNPR3": str(price),
        "ORD_SVR_DVSN_CD": "0",
    }

    print(f"예약 매도 테스트: {ticker} {qty}주 @ ${price}")
    print(f"  tr_id: VTTT3016U (모의투자 예약매도)")
    print()

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, headers=headers("VTTT3016U"), json=body)
        data = resp.json()

    rt_cd = data.get("rt_cd")
    msg = data.get("msg1", "")

    if rt_cd == "0":
        print(f"  예약 매도 성공!")
        print(f"  output: {data.get('output', {})}")
    else:
        print(f"  결과 (rt_cd={rt_cd}): {msg}")

    return data


# === 테스트 실행 ===

print("=" * 60)
print("[테스트 A] 예약주문 - 정수 매수 (1주)")
print("=" * 60)
resv_buy_int = await resv_buy_order("QQQ", qty=1, price=600.0)

await asyncio.sleep(0.1)

print()
print("=" * 60)
print("[테스트 B] 예약주문 - 소수점 매수 (0.5주)")
print("=" * 60)
resv_buy_frac = await resv_buy_order("QQQ", qty=0.5, price=600.0)

await asyncio.sleep(0.1)

print()
print("=" * 60)
print("[테스트 C] 예약주문 - 매도 (1주)")
print("=" * 60)
resv_sell_result = await resv_sell_order("QQQ", qty=1, price=650.0)

[테스트 A] 예약주문 - 정수 매수 (1주)
예약 매수 테스트: QQQ 1주 @ $600.0
  tr_id: VTTT3014U (모의투자 예약매수)
  FT_ORD_QTY = "1"
  body: {'CANO': '50174429', 'ACNT_PRDT_CD': '01', 'OVRS_EXCG_CD': 'NASD', 'PDNO': 'QQQ', 'FT_ORD_QTY': '1', 'FT_ORD_UNPR3': '600.0', 'ORD_SVR_DVSN_CD': '0'}

  결과 (rt_cd=1): 모의투자 예약주문시간을 확인해 주세요.

[테스트 B] 예약주문 - 소수점 매수 (0.5주)
예약 매수 테스트: QQQ 0.5주 @ $600.0
  tr_id: VTTT3014U (모의투자 예약매수)
  FT_ORD_QTY = "0.5"
  body: {'CANO': '50174429', 'ACNT_PRDT_CD': '01', 'OVRS_EXCG_CD': 'NASD', 'PDNO': 'QQQ', 'FT_ORD_QTY': '0.5', 'FT_ORD_UNPR3': '600.0', 'ORD_SVR_DVSN_CD': '0'}

  결과 (rt_cd=1): 주문 수량을 확인해주세요.

[테스트 C] 예약주문 - 매도 (1주)
예약 매도 테스트: QQQ 1주 @ $650.0
  tr_id: VTTT3016U (모의투자 예약매도)

  결과 (rt_cd=1): 초당 거래건수를 초과하였습니다.


## 8-1. 개별 미국 주식 소수점 매수 테스트

ETF(QQQ)가 아닌 개별 주식(AAPL, TSLA 등)으로 소수점 주문이 되는지 확인합니다.

In [17]:
async def test_fractional(ticker, qty, price, exchange="NASD", method="일반주문"):
    """소수점 매수 테스트 (일반주문 vs 예약주문)"""
    if method == "일반주문":
        url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order"
        tr_id = "VTTT1002U"
        body = {
            "CANO": CANO,
            "ACNT_PRDT_CD": ACNT_PRDT_CD,
            "OVRS_EXCG_CD": exchange,
            "PDNO": ticker,
            "ORD_QTY": str(qty),
            "OVRS_ORD_UNPR": str(price),
            "ORD_SVR_DVSN_CD": "0",
            "ORD_DVSN": "00",
        }
    else:
        url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order-resv"
        tr_id = "VTTT3014U"
        body = {
            "CANO": CANO,
            "ACNT_PRDT_CD": ACNT_PRDT_CD,
            "OVRS_EXCG_CD": exchange,
            "PDNO": ticker,
            "FT_ORD_QTY": str(qty),
            "FT_ORD_UNPR3": str(price),
            "ORD_SVR_DVSN_CD": "0",
        }

    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, headers=headers(tr_id), json=body)
        data = resp.json()

    rt_cd = data.get("rt_cd")
    msg = data.get("msg1", "")
    status = "✅ 성공" if rt_cd == "0" else f"❌ {msg}"
    print(f"  {method:>6} | {ticker:<6} | {qty:>5}주 | ${price:>8} | {status}")
    return data

# 예약주문만 재테스트 (rate limit 회피를 위해 1초 간격)
print("예약주문 소수점 재테스트 (1초 간격)")
print(f"{'':>8} | {'종목':<6} | {'수량':>5}  | {'단가':>9} | 결과")
print("-" * 70)

for ticker, price, excd in [("AAPL", 230.0, "NASD"), ("TSLA", 350.0, "NASD"), ("NVDA", 140.0, "NASD"), ("QQQ", 604.0, "NASD")]:
    await test_fractional(ticker, 0.5, price, excd, "예약주문")
    await asyncio.sleep(1.0)  # 1초 대기로 rate limit 회피

예약주문 소수점 재테스트 (1초 간격)
         | 종목     |    수량  |        단가 | 결과
----------------------------------------------------------------------
    예약주문 | AAPL   |   0.5주 | $   230.0 | ❌ 주문 수량을 확인해주세요.
    예약주문 | TSLA   |   0.5주 | $   350.0 | ❌ 주문 수량을 확인해주세요.
    예약주문 | NVDA   |   0.5주 | $   140.0 | ❌ 주문 수량을 확인해주세요.
    예약주문 | QQQ    |   0.5주 | $   604.0 | ❌ 주문 수량을 확인해주세요.


## 9. 🔥 장중 소수점 매수 테스트 (직접 실행용)

**미국 장 시간(KST 23:30~06:00)에 직접 실행하세요!**

아래 셀은 일반주문(VTTT1002U)과 예약주문(VTTT3014U) 두 방식으로
다양한 소수점 수량(0.5, 0.1, 0.01)을 테스트합니다.

장외 시간에는 "장시작전" / "예약주문시간 확인" 에러가 나오므로,
**반드시 장중에 실행**해야 소수점 지원 여부를 정확히 확인할 수 있습니다.

In [19]:
# ⚠️ 토큰이 만료되었을 수 있으니 먼저 재발급
await get_token()
print("토큰 준비 완료 — 아래 테스트를 실행하세요")

토큰 발급 실패: {'error_description': '접근토큰 발급 잠시 후 다시 시도하세요(1분당 1회)', 'error_code': 'EGW00133'}
토큰 준비 완료 — 아래 테스트를 실행하세요


In [20]:
"""
🔥 장중 소수점 매수 종합 테스트
- 방법 1: 일반주문 (VTTT1002U) — ORD_QTY에 소수점
- 방법 2: 예약주문 (VTTT3014U) — FT_ORD_QTY에 소수점
- 종목: QQQ (가장 안전한 ETF)
- 수량: 0.5, 0.1, 0.01, 1 (정수 대조군)
"""

async def fractional_test_standard(ticker, qty, price, exchange="NASD"):
    """일반주문 API로 소수점 매수"""
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order"
    body = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": exchange,
        "PDNO": ticker,
        "ORD_QTY": str(qty),
        "OVRS_ORD_UNPR": str(price),
        "ORD_SVR_DVSN_CD": "0",
        "ORD_DVSN": "00",
    }
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, headers=headers("VTTT1002U"), json=body)
        data = resp.json()
    return data

async def fractional_test_reservation(ticker, qty, price, exchange="NASD"):
    """예약주문 API로 소수점 매수"""
    url = f"{BASE_URL}/uapi/overseas-stock/v1/trading/order-resv"
    body = {
        "CANO": CANO,
        "ACNT_PRDT_CD": ACNT_PRDT_CD,
        "OVRS_EXCG_CD": exchange,
        "PDNO": ticker,
        "FT_ORD_QTY": str(qty),
        "FT_ORD_UNPR3": str(price),
        "ORD_SVR_DVSN_CD": "0",
    }
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(url, headers=headers("VTTT3014U"), json=body)
        data = resp.json()
    return data

# === 현재가 먼저 조회 ===
qqq_now = await get_price("QQQ", "NAS")
limit_price = round(qqq_now * 1.001, 2) if qqq_now else 600.0  # 현재가 약간 위로 지정가
print(f"\n테스트 지정가: ${limit_price}")
print()

# === 테스트 매트릭스 ===
test_cases = [
    # (수량, 설명)
    (1,    "정수 1주 (대조군)"),
    (0.5,  "소수점 0.5주"),
    (0.1,  "소수점 0.1주"),
    (0.01, "소수점 0.01주"),
]

print("=" * 80)
print("📊 일반주문 (VTTT1002U) — ORD_QTY 소수점 테스트")
print("=" * 80)
print(f"{'수량':>8} | {'설명':<20} | {'결과'}")
print("-" * 80)

for qty, desc in test_cases:
    data = await fractional_test_standard("QQQ", qty, limit_price)
    rt_cd = data.get("rt_cd")
    msg = data.get("msg1", "")
    if rt_cd == "0":
        odno = data.get("output", {}).get("ODNO", "?")
        print(f"{qty:>8} | {desc:<20} | ✅ 성공! 주문번호: {odno}")
    else:
        print(f"{qty:>8} | {desc:<20} | ❌ {msg}")
    await asyncio.sleep(1.0)

print()
print("=" * 80)
print("📊 예약주문 (VTTT3014U) — FT_ORD_QTY 소수점 테스트")
print("=" * 80)
print(f"{'수량':>8} | {'설명':<20} | {'결과'}")
print("-" * 80)

for qty, desc in test_cases:
    data = await fractional_test_reservation("QQQ", qty, limit_price)
    rt_cd = data.get("rt_cd")
    msg = data.get("msg1", "")
    if rt_cd == "0":
        odno = data.get("output", {})
        print(f"{qty:>8} | {desc:<20} | ✅ 성공! {odno}")
    else:
        print(f"{qty:>8} | {desc:<20} | ❌ {msg}")
    await asyncio.sleep(1.0)

print()
print("=" * 80)
print("🏁 테스트 완료!")
print("  ✅ 가 나온 소수점 수량이 있으면 → 소수점 매매 가능!")
print("  전부 ❌ 이면 → KIS API는 소수점 매매 미지원 확정")
print("=" * 80)

      종목: DNASQQQ
     현재가: $597.26
    전일종가: $607.69
      고가: $604.14
      저가: $597.05
     거래량: 41867465
     등락률: 0%

테스트 지정가: $597.86

📊 일반주문 (VTTT1002U) — ORD_QTY 소수점 테스트
      수량 | 설명                   | 결과
--------------------------------------------------------------------------------
       1 | 정수 1주 (대조군)          | ❌ 모의투자 장종료 입니다.
     0.5 | 소수점 0.5주             | ❌ MCA 전문바디 구성 중 오류가 발생하였습니다.
     0.1 | 소수점 0.1주             | ❌ MCA 전문바디 구성 중 오류가 발생하였습니다.
    0.01 | 소수점 0.01주            | ❌ MCA 전문바디 구성 중 오류가 발생하였습니다.

📊 예약주문 (VTTT3014U) — FT_ORD_QTY 소수점 테스트
      수량 | 설명                   | 결과
--------------------------------------------------------------------------------
       1 | 정수 1주 (대조군)          | ❌ 모의투자 예약주문시간을 확인해 주세요.
     0.5 | 소수점 0.5주             | ❌ 주문 수량을 확인해주세요.
     0.1 | 소수점 0.1주             | ❌ 주문 수량을 확인해주세요.
    0.01 | 소수점 0.01주            | ❌ 주문 수량을 확인해주세요.

🏁 테스트 완료!
  ✅ 가 나온 소수점 수량이 있으면 → 소수점 매매 가능!
  전부 ❌ 이면 → KIS API는 소수점 매매 미지원 확정


## 10. Raw API 응답 확인

디버깅용: 각 API의 전체 응답을 확인합니다.

In [ ]:
import json

# 잔고 조회 raw 응답
if balance_data:
    print("=== 잔고 조회 raw 응답 ===")
    print(json.dumps(balance_data, indent=2, ensure_ascii=False))